# Multi-component models

A dilute sample holding four non-interacting particle populations scatters as
the plain sum of their form factors, with no structure factor (S(q) = 1):

$$I(q) = \sum_{k=1}^{4} \text{scale}_k \, P_k(q) + \text{background}$$

| moniker | model | what it stands for | where it dominates |
|---|---|---|---|
| `spheres` | `sphere` | large colloids | lowest Q |
| `shells` | `core_shell_sphere` | vesicle-like shells | mid-Q oscillations |
| `rods` | `cylinder` | rod-like aggregates | the $q^{-1}$ region |
| `coils` | `mono_gauss_coil` | free polymer | highest Q |

This notebook:

1. Simulates such a sample with known parameters, so every fitted value can be checked
2. Builds the four-component sum with `set_models()`
3. Decides what to fix and what to fit
4. Fits with a global optimizer and compares the result with the truth
5. Reads the correlations that remain and uses them to choose what to fix next

Every extra component adds amplitudes and sizes that can work against the others
wherever they overlap in Q. A sum of four P(q) can match the data well and still
be wrong. The populations here were chosen to dominate different Q ranges,
which is the best case. Real data is rarely that kind.

In [ ]:
import numpy as np

from sans_fitter import SANSFitter
from sans_fitter.examples import simulate

## 1. Simulate the sample

`set_models()` names parameters after the monikers (`rods_radius`), while
sasmodels itself, which `simulate()` calls, uses `A_`/`B_`/`C_`/`D_` prefixes in
the order the components are given. `to_raw()` translates between the two.

Two choices in the simulated sample matter for the fit:

- **The shells have a core with its own contrast** (`sld_core` differs from
  `sld_solvent`). A hollow shell, whose core matches the solvent, measures only
  scale × thickness², so those two cannot be fitted separately.
- **`mono_gauss_coil` carries its amplitude in `i_zero`**, so its `scale` is
  fixed at 1. Otherwise the pair would be redundant.

In [ ]:
MODELS = {
    'spheres': 'sphere',
    'shells': 'core_shell_sphere',
    'rods': 'cylinder',
    'coils': 'mono_gauss_coil',
}

# Known from composition, held fixed in the fit
FIXED = {
    'spheres_sld': 1.0,
    'spheres_sld_solvent': 6.0,
    'shells_sld_core': 3.0,
    'shells_sld_shell': 1.0,
    'shells_sld_solvent': 6.0,
    'rods_sld': 4.0,
    'rods_sld_solvent': 1.0,
    'coils_scale': 1.0,
}

# What the fit should recover
TRUTH = {
    'spheres_scale': 0.002,
    'spheres_radius': 300.0,
    'shells_scale': 0.005,
    'shells_radius': 40.0,
    'shells_thickness': 15.0,
    'rods_scale': 0.01,
    'rods_radius': 15.0,
    'rods_length': 500.0,
    'coils_i_zero': 1.0,
    'coils_rg': 30.0,
    'background': 0.001,
}


def to_raw(name):
    # Friendly name -> sasmodels A_/B_/C_/D_ name, for simulate()
    if name in ('scale', 'background'):
        return name
    for prefix, moniker in zip('ABCD', MODELS, strict=True):
        if name.startswith(moniker + '_'):
            return f'{prefix}_{name[len(moniker) + 1 :]}'
    raise KeyError(name)


expression = '+'.join(MODELS.values())
truth_raw = {to_raw(k): v for k, v in {**FIXED, **TRUTH}.items()}
data = simulate(expression, qmin=0.003, qmax=0.4, npoints=150, noise=0.02, seed=7, **truth_raw)
print(expression)

## 2. Build the four-component sum

`set_models()` takes the components as `moniker=model` keywords. The simulated
data carries no dQ column, so resolution is switched off explicitly.

In [ ]:
fitter = SANSFitter()
fitter.set_data(data)
fitter.set_resolution('none')
fitter.set_models(**MODELS)
fitter.get_params()

Before fitting, plot each population's contribution at the true parameters.
This shows which Q range constrains which component. It also shows where the
fit will struggle: anything that is never on top of the sum is weakly
determined.

In [ ]:
import plotly.graph_objects as go
from sasdata.dataloader.data_info import Data1D
from sasmodels.core import load_model
from sasmodels.direct_model import DirectModel

truth = {**FIXED, **TRUTH}
q_dense = np.geomspace(data.x.min(), data.x.max(), 1000)  # resolves the sphere fringes
grid = Data1D(x=q_dense, y=np.ones_like(q_dense), dy=np.ones_like(q_dense))
grid.qmin, grid.qmax = q_dense.min(), q_dense.max()
fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=data.x, y=data.y, mode='markers', name='simulated data',
        marker={'size': 4, 'color': 'lightgray'},
        error_y={'type': 'data', 'array': data.dy, 'color': 'lightgray'},
    )
)
for moniker, model in MODELS.items():
    pars = {k[len(moniker) + 1 :]: v for k, v in truth.items() if k.startswith(moniker + '_')}
    y = DirectModel(grid, load_model(model))(**pars, background=0)
    fig.add_trace(go.Scatter(x=q_dense, y=y, mode='lines', name=f'{moniker} ({model})'))
fig.update_xaxes(type='log', title='Q (1/Å)')
fig.update_yaxes(type='log', title='I(Q) (1/cm)', range=[-4, 3])
fig.update_layout(title='Contributions at the true parameters', width=750, height=500)
fig

Each population sits on top of the sum somewhere, except the rods (purple).
They never dominate, and their low-Q plateau, which carries the length, is
buried under the spheres. Expect the fit to struggle with them.

## 3. Decide what to fix, what to fit

- **SLDs are fixed.** In a sum, each component measures only
  scale × contrast², so fitting both scale and SLD is pointless.
- **Sizes and amplitudes are fitted**: 11 free parameters, starting well away
  from the truth.

In [ ]:
# (start, min, max)
START = {
    'spheres_scale': (0.005, 1e-5, 0.1),
    'spheres_radius': (200.0, 100.0, 600.0),
    'shells_scale': (0.01, 1e-5, 0.1),
    'shells_radius': (30.0, 10.0, 100.0),
    'shells_thickness': (10.0, 2.0, 40.0),
    'rods_scale': (0.005, 1e-5, 0.1),
    'rods_radius': (10.0, 3.0, 50.0),
    'rods_length': (300.0, 100.0, 2000.0),
    'coils_i_zero': (0.5, 0.01, 5.0),
    'coils_rg': (20.0, 5.0, 100.0),
    'background': (0.002, 0.0, 0.01),
}

for name, value in FIXED.items():
    fitter.set_param(name, value=value, vary=False)
for name, (value, lo, hi) in START.items():
    fitter.set_param(name, value=value, min=lo, max=hi, vary=True)

## 4. Fit with a global optimizer

Composite models fit with the `bumps` engine only. Use differential evolution
(`method='de'`). A local optimizer started from these guesses can settle in a
minimum where components swap roles: in testing, amoeba returned a good-looking
fit with the sphere and rod sizes exchanged. DE also needs enough steps. With
300 it stopped at χ²/dof ≈ 1.4; 1000 reaches the noise floor. This cell takes
about a minute and a half.

DE is stochastic and draws from NumPy's global random generator, so the seed
makes this run reproducible. Try other seeds: the well-determined parameters
barely move, and the poorly determined ones (below) wander.

In [ ]:
np.random.seed(1)
result = fitter.fit(engine='bumps', method='de', steps=1000)

In [ ]:
def compare(result):
    print(f'{"parameter":<20}{"truth":>10}{"fit":>12}{"error":>10}{"off by":>9}')
    for name, true in TRUTH.items():
        p = result['parameters'][name]
        if p['fixed']:
            print(f'{name:<20}{true:>10.4g}{p["value"]:>12.4g}{"(fixed)":>10}')
            continue
        z = abs(p['value'] - true) / p['stderr']
        print(f'{name:<20}{true:>10.4g}{p["value"]:>12.4g}{p["stderr"]:>10.2g}{z:>8.1f}σ')
    print(f'\nχ²/dof = {result["reduced_chisq"]:.3f}')


compare(result)

In [ ]:
fitter.plot_results(show_components=True)

## 5. Read the correlations

χ²/dof matches the noise, and the fit found the global minimum. Most
parameters come back within about 1σ of the truth. The fit report's
correlation block names what the data does not pin down:

- **`shells_radius` / `shells_thickness` ≈ −1.** The outer radius
  (`radius + thickness`) is well determined. How it splits between core and
  shell is not.
- **`rods_length` is not determined, and its error bar says otherwise.** The
  rods' low-Q Guinier bend sits under the much stronger sphere signal, so only
  their $q^{-1}$ region constrains them. The quoted error is a local
  (Jacobian) estimate. It does not show that χ² stays nearly flat over
  hundreds of Å of rod length. Across seeds the fitted length lands
  anywhere from about 250 to 900 Å.
- **`coils_i_zero` / `coils_rg`.** The coil is visible only at high Q, where
  it outlasts the other components.

The remedy is the same each time: **fix what you know from somewhere else.**
Suppose the rod length is known from microscopy. Fix it and refit. Starting
from the DE solution, a fast local optimizer is now enough. Compare χ²/dof
before and after.

In [ ]:
outer = fitter.params['shells_radius']['value'] + fitter.params['shells_thickness']['value']
print(f'shell outer radius: fit {outer:.1f}, truth {TRUTH["shells_radius"] + TRUTH["shells_thickness"]:.1f}')

In [ ]:
fitter.set_param('rods_length', value=500.0, vary=False)
result_fixed = fitter.fit(engine='bumps', method='amoeba')
compare(result_fixed)

χ²/dof barely changes when the rod length is fixed at its true value, and
the other parameters hardly move. The data never contained the rod length.
Fixing it from independent knowledge costs nothing and removes a parameter
that would otherwise be reported with a misleading error bar.

## Summary

- `set_models(moniker=model, ...)` combines any number of form factors. Each
  component's parameters get the moniker as prefix, and the fit uses `bumps`.
- Fix contrast (SLDs), and fix redundant amplitudes such as a model's own
  `i_zero` against its `scale`.
- Fit with `method='de'` and enough steps. Confirm χ²/dof matches the noise
  before trusting any parameter.
- Correlations near ±1 in the fit report mark parameters the data cannot
  separate. Fix one of each such pair from independent knowledge, then refit.
